# Week 3 exercise — open-source models, locally

Week 3 is about Hugging Face: **tokenizers**, **pipelines**, and running **open-source
models**. The Day-5 meeting-minutes project uses Whisper + Llama-8B on a Colab GPU; here
is a minimal version that runs **locally on CPU** — compare tokenizers, then turn a
meeting transcript into minutes with an open-source summarizer.

In [1]:
from transformers import AutoTokenizer, pipeline

## 1. Tokenizers
Different models split the same text differently. Tokenizers are tiny (no model weights),
so this runs instantly. (Uses ungated models — no Hugging Face login needed.)

In [2]:
text = "Tokenization turns text into integers a model can process."
for name in ["gpt2", "bert-base-uncased", "Qwen/Qwen2.5-0.5B-Instruct"]:
    tok = AutoTokenizer.from_pretrained(name)
    ids = tok.encode(text)
    print(f"{name:<28} {len(ids):>3} tokens  ->  {tok.convert_ids_to_tokens(ids)[:8]} ...")

gpt2                          11 tokens  ->  ['Token', 'ization', 'Ġturns', 'Ġtext', 'Ġinto', 'Ġintegers', 'Ġa', 'Ġmodel'] ...
bert-base-uncased             13 tokens  ->  ['[CLS]', 'token', '##ization', 'turns', 'text', 'into', 'integers', 'a'] ...


Qwen/Qwen2.5-0.5B-Instruct    11 tokens  ->  ['Token', 'ization', 'Ġturns', 'Ġtext', 'Ġinto', 'Ġintegers', 'Ġa', 'Ġmodel'] ...


## 2. Meeting minutes from an open-source model
A local summarization pipeline (open weights, CPU) condenses a transcript into minutes.

In [3]:
transcript = """
Alex: Welcome everyone. The goal today is to lock the launch plan for the mobile app.
Maria: Beta testing is done. We found two minor bugs, both fixed and verified yesterday.
Sam: Marketing is ready. We'll send the announcement email on Friday morning.
Alex: Great. Let's ship to the app stores Thursday so Friday's email points to a live app.
Maria: I'll submit the build Thursday and monitor the review queue.
Sam: I'll prepare the social posts and schedule them for Friday.
Alex: Perfect. We meet again Monday to review launch metrics.
"""

summarizer = pipeline("summarization", model="Falconsai/text_summarization")
minutes = summarizer(transcript, max_length=80, min_length=25, do_sample=False)[0]["summary_text"]
print("MEETING MINUTES\n" + "-" * 14 + "\n" + minutes)

Device set to use cpu


Both `max_new_tokens` (=256) and `max_length`(=80) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


MEETING MINUTES
--------------
Alex: Great. Let's ship to the app stores Thursday so Friday's email points to a live app . Sam: Marketing is ready. We'll send the announcement email on Friday morning .


**Full version (Colab GPU):** transcribe real audio with `openai/whisper`, then generate
structured minutes with a quantized `meta-llama/Llama-3.1-8B-Instruct` — see Day 5.